# Week 1: Transformer Architecture

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week01-transformers.html

**How to use:** run the cells from the top (Runtime → Run all, or Shift+Enter cell by cell). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need a key: add it with the 🔑 Secrets panel on the left, or set `os.environ["OPENAI_API_KEY"] = "..."` in a cell. Examples that use Ollama need a local Ollama server and will not work in Colab. For GPU examples choose Runtime → Change runtime type → T4 GPU.

In [ ]:
# Imports that some of the lesson's code fragments assume
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

## Self-attention implementation (PyTorch)

In [ ]:
import torch
import torch.nn as nn
import math


class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert self.head_dim * heads == embed_size, "Embed size must be divisible by heads"

        # Linear projections for queries, keys, values and the output
        self.q_linear = nn.Linear(embed_size, embed_size)
        self.k_linear = nn.Linear(embed_size, embed_size)
        self.v_linear = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # Project and split into heads: (batch, seq, heads, head_dim) -> (batch, heads, seq, head_dim)
        q = self.q_linear(query).view(batch_size, -1, self.heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.k_linear(key).view(batch_size, -1, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.v_linear(value).view(batch_size, -1, self.heads, self.head_dim).permute(0, 2, 1, 3)

        # Scaled dot-product scores: (batch, heads, seq_q, seq_k)
        scores = torch.matmul(q, k.permute(0, 1, 3, 2)) / math.sqrt(self.head_dim)

        # Optional mask (used in the decoder)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-1e20"))

        # Attention weights
        attention = torch.softmax(scores, dim=-1)

        # Weighted sum of values: (batch, heads, seq, head_dim)
        out = torch.matmul(attention, v)

        # Merge heads: (batch, seq, heads, head_dim) -> (batch, seq, embed_size)
        out = out.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.embed_size)

        return self.fc_out(out)

## Using Transformers with libraries (Hugging Face)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "Understanding transformers is fascinating."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
pooled_output = outputs.pooler_output

print(f"Hidden state shape: {last_hidden_state.shape}")  # [batch_size, sequence_length, hidden_size]
print(f"Pooled output shape: {pooled_output.shape}")     # [batch_size, hidden_size]

## Run in your browser

**Scaled dot-product attention for a 4-token sentence (NumPy)**

In [ ]:
import numpy as np

tokens = ["The", "cat", "sat", "down"]
Q = np.array([[1, 0, 1], [0, 2, 0], [1, 1, 0], [0, 0, 1]], dtype=float)
K = np.array([[1, 0, 0], [0, 2, 1], [0, 1, 1], [1, 0, 1]], dtype=float)
V = np.array([[1, 0], [0, 1], [1, 1], [0, 0]], dtype=float)

d_k = Q.shape[1]
scores = Q @ K.T / np.sqrt(d_k)                      # (4, 4)
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)         # softmax per row
output = weights @ V                                  # (4, 2)

np.set_printoptions(precision=2, suppress=True)
print("attention weights (rows = queries):")
for t, row in zip(tokens, weights):
    print(f"  {t:>5}", row)
print("\noutput vectors:\n", output)
print("\nEach row of weights sums to 1:", weights.sum(axis=1))

**Sinusoidal positional encoding**

In [ ]:
import numpy as np

def positional_encoding(n_positions, d_model):
    pos = np.arange(n_positions)[:, None]
    i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)
    return np.where(i % 2 == 0, np.sin(angle), np.cos(angle))

pe = positional_encoding(6, 8)
np.set_printoptions(precision=3, suppress=True)
print(pe)
# Nearby positions get similar vectors; the dot product falls off with distance
print("\nsimilarity of position 0 with positions 0..5:", pe @ pe[0])

**Matrix multiplication by hand vs NumPy**

In [ ]:
import numpy as np

A = np.array([[-1, 4, 2], [4, -2, 3]])      # 2 x 3
B = np.array([[0, -2], [-2, 4], [4, 1]])    # 3 x 2

C = np.zeros((A.shape[0], B.shape[1]), dtype=int)
for i in range(A.shape[0]):
    for j in range(B.shape[1]):
        terms = [A[i, k] * B[k, j] for k in range(A.shape[1])]
        C[i, j] = sum(terms)
        print(f"c[{i+1},{j+1}] = " + " + ".join(f"({A[i,k]})({B[k,j]})" for k in range(A.shape[1])) + f" = {C[i, j]}")

print("\nby hand:\n", C)
print("A @ B:\n", A @ B)
print("same result:", np.array_equal(C, A @ B))

## Test code for the course environment

**Scaled dot-product attention from scratch (NumPy)**

In [ ]:
# 1. Scaled dot-product attention from scratch (NumPy)
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores)
    return weights @ V, weights

rng = np.random.default_rng(0)
seq, d = 5, 8
X = rng.normal(size=(seq, d))
Wq, Wk, Wv = (rng.normal(size=(d, d)) * 0.3 for _ in range(3))
out, w = attention(X @ Wq, X @ Wk, X @ Wv)
print("attention weights (rows sum to 1):\n", w.round(3))
print("row sums:", w.sum(axis=1).round(6))
print("output shape:", out.shape)

**Check the PyTorch SelfAttention module from the lesson**

In [ ]:
# 2. Check the PyTorch SelfAttention module from the lesson
import torch
from torch import nn
import math

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size, self.heads = embed_size, heads
        self.head_dim = embed_size // heads
        assert self.head_dim * heads == embed_size
        self.q_linear = nn.Linear(embed_size, embed_size)
        self.k_linear = nn.Linear(embed_size, embed_size)
        self.v_linear = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, query, key, value, mask=None):
        B = query.shape[0]
        q = self.q_linear(query).view(B, -1, self.heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.k_linear(key).view(B, -1, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.v_linear(value).view(B, -1, self.heads, self.head_dim).permute(0, 2, 1, 3)
        scores = torch.matmul(q, k.permute(0, 1, 3, 2)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-1e20"))
        att = torch.softmax(scores, dim=-1)
        out = torch.matmul(att, v).permute(0, 2, 1, 3).contiguous().view(B, -1, self.embed_size)
        return self.fc_out(out)

x = torch.randn(2, 7, 64)               # batch=2, seq=7, embed=64
sa = SelfAttention(embed_size=64, heads=8)
print(sa(x, x, x).shape)                # torch.Size([2, 7, 64])

# causal mask: position i may only attend to j <= i
mask = torch.tril(torch.ones(7, 7))[None, None]   # (1,1,7,7)
print(sa(x, x, x, mask=mask).shape)

**Sinusoidal positional encoding heatmap**

In [ ]:
# 3. Sinusoidal positional encoding heatmap
import numpy as np
import matplotlib.pyplot as plt

def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

pe = positional_encoding(50, 64)
plt.imshow(pe, cmap="RdBu", aspect="auto")
plt.xlabel("dimension"); plt.ylabel("position"); plt.colorbar()
plt.title("Sinusoidal positional encoding")
plt.show()

**Run BERT with Hugging Face and inspect output shapes**

In [ ]:
# 4. Run BERT with Hugging Face and inspect output shapes
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

inputs = tokenizer("Understanding transformers is fascinating.", return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Hidden state shape:", outputs.last_hidden_state.shape)  # [1, seq_len, 768]
print("Pooled output shape:", outputs.pooler_output.shape)     # [1, 768]